# Add Delta Tables to Genie Space

Adds the three Delta tables created in **notebook 3** to the Genie space **build-con-mp** created in **notebook 1**. After running this notebook, you can ask natural-language questions about the EHR, lung cancer images, and parsed clinical notes in that Genie space.

**Tables added:**
- `melissap.melissa_pang.ehr`
- `melissap.melissa_pang.lung_cancer_images`
- `melissap.melissa_pang.parsed_clinical_notes`

**Prerequisites:** Run notebook 1 (create Genie space) and notebook 3 (create the three Delta tables).

In [1]:
# Configuration — must match notebook 1 (Genie space) and notebook 3 (tables)
GENIE_SPACE_TITLE = "build-con-mp"
CATALOG = "melissap"
SCHEMA = "melissa_pang"

EHR_TABLE = f"{CATALOG}.{SCHEMA}.ehr"
IMAGES_TABLE = f"{CATALOG}.{SCHEMA}.lung_cancer_images"
NOTES_TABLE = f"{CATALOG}.{SCHEMA}.parsed_clinical_notes"

TABLE_IDENTIFIERS = [EHR_TABLE, IMAGES_TABLE, NOTES_TABLE]

In [2]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient(profile="DEFAULT")

In [3]:
# Find the Genie space by title (created in notebook 1)
response = w.genie.list_spaces(page_size=100)
spaces = response.spaces or []
match = next((s for s in spaces if (s.title or "").strip() == GENIE_SPACE_TITLE), None)

if not match:
    raise RuntimeError(
        f"Genie space '{GENIE_SPACE_TITLE}' not found. Run notebook 1 (setup_volume_and_genie) first to create it."
    )

space_id = match.space_id
print(f"Found Genie space: {match.title} (id: {space_id})")

Found Genie space: build-con-mp (id: 01f118aedb38167081b3a7a8edaebca5)


In [5]:
# Get current space config; Genie expects serialized_space with version 2 and data_sources.tables
space = w.genie.get_space(space_id=space_id, include_serialized_space=True)
current = space.serialized_space or "{}"

try:
    payload = json.loads(current)
except json.JSONDecodeError:
    payload = {}

if not isinstance(payload, dict):
    payload = {}

# API expects version 2 and data_sources.tables as list of { "identifier": "catalog.schema.table" }
payload["version"] = 2
if "data_sources" not in payload or not isinstance(payload["data_sources"], dict):
    payload["data_sources"] = {}
payload["data_sources"]["tables"] = [{"identifier": t} for t in TABLE_IDENTIFIERS]
if "metric_views" not in payload["data_sources"]:
    payload["data_sources"]["metric_views"] = []
new_serialized = json.dumps(payload)

w.genie.update_space(space_id=space_id, serialized_space=new_serialized)
print(f"Updated Genie space with {len(TABLE_IDENTIFIERS)} tables:")
for t in TABLE_IDENTIFIERS:
    print(f"  - {t}")
print("\nOpen in workspace: AI/BI Genie > find space by title '" + GENIE_SPACE_TITLE + "'")

Updated Genie space with 3 tables:
  - melissap.melissa_pang.ehr
  - melissap.melissa_pang.lung_cancer_images
  - melissap.melissa_pang.parsed_clinical_notes

Open in workspace: AI/BI Genie > find space by title 'build-con-mp'
